In [ ]:
import json
import os
import time
from google import genai
from google.genai import types

# --- Configuração da API Gemini ---
# IMPORTANTE: Para segurança, é recomendado usar variáveis de ambiente.
API_KEY = "SUA_API_AQUI" 

# Inicializa o cliente Gemini, conforme o padrão compatível
try:
    client = genai.Client(api_key=API_KEY)
except Exception as e:
    print(f"Falha ao inicializar o cliente Gemini. Verifique sua API Key. Erro: {e}")
    exit()

# Nome do modelo Gemini a ser utilizado (o mesmo usado na extração sem RAG, no script-unificado.ipynb)
MODELO_GEMINI = "gemini-2.5-flash"

# --- TRATAMENTO DE ERROS DE API ---
# Chamadas que falham são repetidas com espera crescente. Se todas as tentativas falharem, a resposta é
# gravada com o prefixo ERRO_API: ela não é enviada para extração, é contada à parte na avaliação
# e é refeita automaticamente na próxima execução do script.
ERRO_API = "erro_api"
TENTATIVAS_API = 3
ESPERA_INICIAL_S = 5  # dobra a cada nova tentativa (5s, 10s, ...)

def eh_erro_api(texto):
    return str(texto).startswith(ERRO_API)

def chamar_com_retentativas(chamada, descricao):
    """Executa 'chamada' (função sem argumentos que retorna o texto da resposta ou levanta exceção).
    Retorna o texto, ou f"{ERRO_API}: <motivo>" se todas as tentativas falharem."""
    for tentativa in range(1, TENTATIVAS_API + 1):
        try:
            return chamada()
        except Exception as e:
            print(f"   [{descricao}] Falha na tentativa {tentativa}/{TENTATIVAS_API}: {e}")
            if tentativa < TENTATIVAS_API:
                time.sleep(ESPERA_INICIAL_S * 2 ** (tentativa - 1))
            else:
                return f"{ERRO_API}: {e}"

In [ ]:

# --- 2. ARQUIVOS ---
# Pares (entrada, saída): saídas do 1. script_rag_v3_html.ipynb -> arquivos usados pelo junta_resultados.py
ARQUIVOS = [
    ("resultados_rag_phi-4-mini_v3.json", "resultados_rag_phi-4-mini_v3_extraido.json"),
    ("resultados_rag_gemini-2.5-flash_v3.json", "resultados_rag_gemini-2.5-flash_v3_extraido.json"),
]

linha_inicio = 1
linha_fim = -1  # Use -1 para processar até o final do arquivo
# ------------------------------------

# --- Dicionário de Prompts de Extração ---

PROMPTS_EXTRACAO = {
    "PERGUNTA01": (
        "Sua tarefa é extrair um valor de memória Flash de um texto. "
        "Retorne apenas o valor numérico com sua unidade (B, KB, MB, GB), sem qualquer texto adicional. "
        "Exemplos de entrada e saída esperada:\n"
        "- Entrada: 'O microcontrolador possui 64 Kbytes de memória flash.' -> Saída: '64KB'\n"
        "- Entrada: 'A memória é de 2 megabytes.' -> Saída: '2MB'\n"
        "- Entrada: 'Não foi possível encontrar a informação.' -> Saída: 'não sei'"
    ),
    "PERGUNTA02": (
        "Sua tarefa é extrair uma velocidade de clock de um texto. "
        "Retorne apenas o valor numérico com sua unidade (Hz, KHz, MHz, GHz), sem qualquer texto adicional. "
        "Exemplos de entrada e saída esperada:\n"
        "- Entrada: 'A velocidade de operação é de até 72 MHz.' -> Saída: '72MHz'\n"
        "- Entrada: 'O clock principal é 16 megahertz.' -> Saída: '16MHz'\n"
        "- Entrada: 'A especificação não informa a velocidade.' -> Saída: 'não sei'"
    ),
    "PERGUNTA03": (
        "Sua tarefa é extrair o número total de portas de Entrada/Saída (I/O) de um texto. "
        "Retorne apenas o número total, sem texto ou unidades. Some os valores se necessário. "
        "Exemplos de entrada e saída esperada:\n"
        "- Entrada: 'Este modelo possui 50 pinos de I/O.' -> Saída: '50'\n"
        "- Entrada: 'Ele tem 32 entradas e 16 saídas.' -> Saída: '48'\n"
        "- Entrada: 'Conta com 24 portas de uso geral.' -> Saída: '24'\n"
        "- Entrada: 'Não encontrei o número de I/Os.' -> Saída: 'não sei'"
    ),
    "PERGUNTA04": (
        "Analise o texto a seguir e determine se o microcontrolador possui comunicação CAN bus. "
        "Responda estritamente com 'sim', 'não', ou 'não sei'."
    ),
    "PERGUNTA05": (
        "Analise o texto a seguir e determine se o microcontrolador possui comunicação I2C. "
        "Responda estritamente com 'sim', 'não', ou 'não sei'."
    ),
    "PERGUNTA06": (
        "Analise o texto a seguir e determine se o microcontrolador possui comunicação Ethernet. "
        "Responda estritamente com 'sim', 'não', ou 'não sei'."
    )

}

def extrair_valor_com_gemini(texto_completo: str, prompt_template: str) -> str:
    """Usa o modelo Gemini para extrair um valor de um texto com base em um prompt."""
    # Respostas vazias ou que falharam na geração não são enviadas para extração
    if not texto_completo or texto_completo.strip() == "" or eh_erro_api(texto_completo):
        return ERRO_API

    prompt_final = f"{prompt_template}\n\nAnalise o seguinte texto:\n---\n{texto_completo}\n---\nValor extraído:"

    def chamada():
        # Usando o padrão client.models.generate_content
        response = client.models.generate_content(
            model=MODELO_GEMINI,
            contents=prompt_final,
            config=types.GenerateContentConfig(
                temperature=0.1,
                thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
        )
        if not response.text:
            raise ValueError("a API não retornou texto")
        return response.text.strip()

    valor = chamar_com_retentativas(chamada, "Gemini (Extração)")
    return ERRO_API if eh_erro_api(valor) else valor

def salvar_dados_json(dados: list, caminho_arquivo: str):
    """Função helper para salvar a lista de dados em um arquivo JSON."""
    try:
        with open(caminho_arquivo, 'w', encoding='utf-8') as f:
            json.dump(dados, f, indent=4, ensure_ascii=False)
    except IOError as e:
        print(f"ERRO CRÍTICO: Não foi possível salvar o arquivo {caminho_arquivo}. Erro: {e}")

def gerar_extracao_json(arquivo_entrada: str, arquivo_saida: str, inicio: int, fim: int):
    """Lê um arquivo JSON, GERA o campo 'valor_extraido' e salva incrementalmente."""
    
    # --- 1. Carregar dados de ENTRADA ---
    try:
        with open(arquivo_entrada, 'r', encoding='utf-8') as f:
            dados_completos = json.load(f)
    except FileNotFoundError:
        print(f"Erro: Arquivo de entrada '{arquivo_entrada}' não encontrado.")
        return
    except json.JSONDecodeError:
        print(f"Erro: O conteúdo de '{arquivo_entrada}' não é um JSON válido.")
        return
    
    total_registros_original = len(dados_completos)

    # --- 2. Montar dados de SAÍDA a partir da ENTRADA ---
    # A entrada é sempre a base, para que respostas refeitas no RAG (ex: após erro de API) sejam incorporadas.
    # Se já existir um arquivo de saída, as extrações anteriores são reaproveitadas para as respostas que não mudaram.
    dados_para_salvar = dados_completos
    if os.path.exists(arquivo_saida):
        print(f"Arquivo de saída '{arquivo_saida}' encontrado. Reaproveitando extrações anteriores...")
        try:
            with open(arquivo_saida, 'r', encoding='utf-8') as f:
                dados_anteriores = json.load(f)
        except json.JSONDecodeError:
            print(f"Erro: Arquivo de saída '{arquivo_saida}' está corrompido. Renomeie-o e tente novamente.")
            return

        anteriores = {}
        for item in dados_anteriores:
            chave_item = (item.get("linha_excel"), item.get("coluna_pergunta"), item.get("pergunta"))
            for resposta in item.get("respostas_modelos", []):
                anteriores[chave_item + (resposta.get("modelo"),)] = resposta

        reaproveitadas = 0
        for item in dados_para_salvar:
            chave_item = (item.get("linha_excel"), item.get("coluna_pergunta"), item.get("pergunta"))
            for resposta in item.get("respostas_modelos", []):
                anterior = anteriores.get(chave_item + (resposta.get("modelo"),))
                if anterior and "valor_extraido" in anterior and anterior.get("resposta_completa") == resposta.get("resposta_completa"):
                    resposta["valor_extraido"] = anterior["valor_extraido"]
                    if "modelo_extracao" in anterior:
                        resposta["modelo_extracao"] = anterior["modelo_extracao"]
                    reaproveitadas += 1
        print(f"{reaproveitadas} extrações anteriores reaproveitadas.")
    else:
        print(f"Arquivo de saída '{arquivo_saida}' não encontrado. Criando a partir da entrada.")

    # --- 3. Ajustar e Validar Intervalo ---
    if fim == -1:
        fim = total_registros_original
    
    if inicio < 1 or fim > total_registros_original or inicio > fim:
        print(f"Erro: Intervalo de linhas inválido. Por favor, defina um valor entre 1 e {total_registros_original}.")
        return

    # Ajuste para índice 0
    indices_para_processar = range(inicio - 1, fim)

    print(f"Iniciando a EXTRAÇÃO de registros de {inicio} a {fim} (total de {total_registros_original} registros no arquivo).")
    print(f"Salvando resultados em: {arquivo_saida}")

    # --- 4. Loop Principal de Processamento ---
    for index in indices_para_processar:
        # Pega o item da estrutura de SAÍDA (que está sendo modificada)
        item = dados_para_salvar[index]
        
        coluna_pergunta = item.get("coluna_pergunta")
        prompt_base = PROMPTS_EXTRACAO.get(coluna_pergunta)

        print(f"\n--- Processando Registro {index + 1}/{total_registros_original} (Pergunta: '{coluna_pergunta}') ---")

        if not prompt_base:
            print(f"Aviso: Pulando. Motivo: 'coluna_pergunta' ('{coluna_pergunta}') sem prompt definido.")
            continue

        # Itera sobre a lista de respostas de modelos (ex: llama, gemini, etc.)
        for resposta in item.get("respostas_modelos", []):
            modelo_nome = resposta.get("modelo", "N/A")

            # Reaproveita extrações válidas feitas com este mesmo modelo (permite retomar e refazer só os erros)
            if (resposta.get("modelo_extracao") == MODELO_GEMINI and "valor_extraido" in resposta
                    and not eh_erro_api(resposta["valor_extraido"])):
                continue
            
            # --- LÓGICA DE EXTRAÇÃO ---
            # 1. Pega a resposta completa que servirá de base
            resposta_completa = resposta.get("resposta_completa")
            
            # 2. Pega o valor extraído antigo (APENAS PARA LOG)
            valor_extraido_antigo = resposta.get("valor_extraido", "N/A (inexistente)")
            
            print(f"   - Processando modelo: {modelo_nome}")
            print(f"     Valor antigo: '{valor_extraido_antigo}'")

            # 3. Chama a API para extrair o novo valor
            novo_valor = extrair_valor_com_gemini(resposta_completa, prompt_base)
            
            print(f"     Novo valor:   '{novo_valor}'")
            
            # 4. Define o novo valor no campo "valor_extraido"
            resposta["valor_extraido"] = novo_valor
            resposta["modelo_extracao"] = MODELO_GEMINI
            
            # Pausa para evitar limites de taxa da API (QPM - Queries Per Minute)
            # 2 segundos é um valor seguro. Se tiver 60 QPM, pode usar 1 segundo.
            time.sleep(2) 

        # Salva o progresso no arquivo de saída APÓS processar CADA item
        salvar_dados_json(dados_para_salvar, arquivo_saida)
        print(f"Progresso salvo em '{arquivo_saida}' após processar o registro {index + 1}.")
    
    erros = sum(eh_erro_api(r.get("valor_extraido")) for item in dados_para_salvar for r in item.get("respostas_modelos", []))
    if erros:
        print(f"\nATENÇÃO: {erros} respostas estão com erro de API. Rode esta célula novamente para refazer as extrações que falharam.")
    print(f"\nProcessamento do intervalo [{inicio}-{fim}] concluído!")

# --- EXECUÇÃO DO SCRIPT ---
if __name__ == "__main__":
    for arquivo_entrada, arquivo_dados_saida in ARQUIVOS:
        gerar_extracao_json(
            arquivo_entrada, 
            arquivo_dados_saida, 
            linha_inicio, 
            linha_fim
        )

In [ ]:
"""
import json
import plotly.graph_objects as go
import plotly.express as px

# Nome do arquivo JSON
arquivo_json = "resultados_rag_gpt-oss_com_instrucao.json"

# Carrega os dados do arquivo JSON
try:
    with open(arquivo_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print(f"Erro: O arquivo '{arquivo_json}' não foi encontrado.")
    exit()
except json.JSONDecodeError:
    print(f"Erro: Não foi possível decodificar o arquivo '{arquivo_json}'. Verifique se ele é um JSON válido.")
    exit()

# Inicializa dicionários para armazenar os dados de desempenho
model_performance = {}

# Itera sobre cada pergunta no arquivo
for question in data:
    # Corrige para evitar erro se 'resposta_verdadeira' for None
    resposta_verdadeira = question.get("resposta_verdadeira")
    if resposta_verdadeira is not None:
        true_answer = str(resposta_verdadeira).lower().strip().replace('.', '')
    else:
        true_answer = ""
    model_responses = question["respostas_modelos"]

    for response in model_responses:
        model_name = response["modelo"]
        extracted_value = response["valor_extraido"].lower().strip().replace('.', '')

        if model_name not in model_performance:
            model_performance[model_name] = {"correct": 0, "incorrect": 0, "nao_sei": 0, "total": 0}

        if extracted_value == "não sei" or extracted_value == "nãosei" or extracted_value == "NÃO SEI" or extracted_value == "NAOSEI":
            model_performance[model_name]["nao_sei"] += 1
        elif extracted_value == true_answer:
            model_performance[model_name]["correct"] += 1
        else:
            model_performance[model_name]["incorrect"] += 1

        model_performance[model_name]["total"] += 1

# Imprime o desempenho de cada modelo
print("---")
print("Performance dos Modelos:")
# ...existing code...
for model, stats in model_performance.items():
    correct = stats["correct"]
    incorrect = stats["incorrect"]
    nao_sei = stats["nao_sei"]
    total = stats["total"]
    correct_pct = (correct / total) * 100 if total > 0 else 0
    incorrect_pct = (incorrect / total) * 100 if total > 0 else 0
    nao_sei_pct = (nao_sei / total) * 100 if total > 0 else 0
    print(f"Modelo: {model}")
    print(f"  Certas: {correct} ({correct_pct:.2f}%) | Erradas: {incorrect} ({incorrect_pct:.2f}%) | Não sei: {nao_sei} ({nao_sei_pct:.2f}%)")
    print(f"  Total de respostas: {total}")
    print("-" * 30)
# ...existing code...

# Prepara os dados para o gráfico
models = list(model_performance.keys())

# Valores absolutos
corrects_abs = [model_performance[m]["correct"] for m in models]
incorrects_abs = [model_performance[m]["incorrect"] for m in models]
nao_seis_abs = [model_performance[m]["nao_sei"] for m in models]

# Valores percentuais
corrects_pct = [corrects_abs[i] / model_performance[m]["total"] * 100 if model_performance[m]["total"] > 0 else 0 for i, m in enumerate(models)]
incorrects_pct = [incorrects_abs[i] / model_performance[m]["total"] * 100 if model_performance[m]["total"] > 0 else 0 for i, m in enumerate(models)]
nao_seis_pct = [nao_seis_abs[i] / model_performance[m]["total"] * 100 if model_performance[m]["total"] > 0 else 0 for i, m in enumerate(models)]

# Cria o gráfico de barras empilhadas com Plotly
fig = go.Figure()

# Adiciona a barra de respostas corretas
fig.add_trace(go.Bar(
    x=models,
    y=corrects_pct,
    name='Corretas',
    marker_color='#5CB85C',
    text=[f'{corrects_abs[i]} ({p:.1f}%)' for i, p in enumerate(corrects_pct)],
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='white', size=11)
))

# Adiciona a barra de respostas erradas
fig.add_trace(go.Bar(
    x=models,
    y=incorrects_pct,
    name='Erradas',
    marker_color='#D9534F',
    text=[f'{incorrects_abs[i]} ({p:.1f}%)' for i, p in enumerate(incorrects_pct)],
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='white', size=11)
))

# Adiciona a barra de "não sei"
fig.add_trace(go.Bar(
    x=models,
    y=nao_seis_pct,
    name='Não sei',
    marker_color='#A9A9A9',
    text=[f'{nao_seis_abs[i]} ({p:.1f}%)' for i, p in enumerate(nao_seis_pct)],
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='black', size=11)
))

# Configura o layout do gráfico
fig.update_layout(
    barmode='stack',
    title={
        'text': 'Desempenho dos Modelos (Corretas, Erradas, Não sei)',
        'y':0.9,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title='Modelo',
    yaxis_title='Percentual de Respostas (%)',
    legend_title='Legenda',
    font=dict(
        family="Arial, sans-serif",
        size=12,
        color="#7f7f7f"
    ),
    hovermode="x unified",
    template='plotly_white'
)

# Exibe e salva o gráfico interativo
fig.write_html("performance-phi-4.html")
print("---")
print("Gráfico de desempenho empilhado gerado e salvo como 'performance_modelos_empilhado.html'!")
print("---")
"""